# Normalize the GUD conllu files to Cypriotic Greek

Install dependencies 
- python-docx
- pandas
- openpyxl

Initialize the normalizer

In [1]:
import sys

sys.path.append("CyGr-Normalizer/code/")
from ProcessCyGr import *

In [2]:
# change to your path
PATH = '/Users/maria/Documents/DPMS/cypriotic/cypriot-treebank-project/CyGr-Normalizer/'

rules = ['1-smooth.xlsx', '2-corrections.xlsx', '3-restore.xlsx']
# the first path is just a dummy to initialize the normalizer
far_tool = ProcessCyGr(PATH, PATH + 'spelling-normalizer/', rules)

Tokenize the conllu files 

In [3]:
from conllu import parse_incr, TokenList

def format_conllu_file(input_filepath, output_filepath):
    """
    Reads a CoNLL-U file, removes multi-word token declarations, cleans 
    the MISC column of 'SpaceAfter=No', rebuilds the text metadata, 
    and writes the output to a new file, preserving all original metadata (e.g., sent_id).
    """
    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', encoding='utf-8') as outfile:
        
        # parse_incr reads the file lazily sentence by sentence
        for sentence in parse_incr(infile):
            
            # --- 1. Delete multiword token declarations (e.g., 13-14) ---
            filtered_tokens = []
            for token in sentence:
                if isinstance(token["id"], tuple) and token["id"][1] == "-":
                    continue  # Skip the MWT declaration
                filtered_tokens.append(token)
            
            # --- 2. Rebuild the text and clean the MISC column ---
            forms = []

            for token in filtered_tokens:
                # Collect standard token forms for the metadata text
                if isinstance(token["id"], int):
                    forms.append(token["form"])
                
                # Process the MISC dictionary natively provided by the conllu package
                if token["misc"] is not None:
                    if token["misc"].get("SpaceAfter") == "No":
                        # Delete the SpaceAfter key
                        del token["misc"]["SpaceAfter"]
                        
                        # If deleting it leaves the dictionary empty, set it to None.
                        if len(token["misc"]) == 0:
                            token["misc"] = None

            # --- 3. Create a new TokenList to safely preserve ALL metadata ---
            # We pass in the modified tokens and explicitly attach the original metadata
            new_sentence = TokenList(filtered_tokens, metadata=sentence.metadata)

            # Update the text metadata specifically (sent_id remains untouched)
            new_sentence.metadata["text"] = " ".join(forms)

            # --- 4. Write the modified sentence back to the file ---
            outfile.write(new_sentence.serialize())





In [4]:
format_conllu_file("el_gud-ud-train.conllu", "el_gud-ud-train-tokenized.conllu")
format_conllu_file("el_gud-ud-test.conllu", "el_gud-ud-test-tokenized.conllu")

Steps to follow
1. normalize the text field
2. replace the forms column with the normalized tokens
3. normalize the lemmas

In [8]:
from conllu import parse_incr

def normalize_conllu_fields(input_filepath, output_filepath):
    """
    1. Normalizes the # text metadata field.
    2. Verifies token counts match between the normalized text and the forms.
    3. Updates the token forms with the normalized words.
    4. Normalizes the lemmas.
    """
    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', encoding='utf-8') as outfile:
        
        for sentence in parse_incr(infile):
            
            # --- 1. Apply normalization to the text field ---
            original_text = sentence.metadata.get("text", "")
            
            if original_text:
                normalized_text = far_tool.normalize_text(original_text)
                normalized_text = normalized_text.replace('\u037e',';').strip()
                normalized_text = normalized_text.replace("σ’", "σ'")
                normalized_text = normalized_text.replace("’", "'")
                # Update the metadata 
                sentence.metadata["text"] = normalized_text
                # Split by whitespace to get the individual tokens
                normalized_tokens = normalized_text.split(" ")
            else:
                normalized_tokens = []
            
            # Isolate the standard tokens that contain the forms
            standard_tokens = [t for t in sentence if isinstance(t["id"], int)]
            
            # --- 2. Check that normalized text token count matches form count ---
            if len(normalized_tokens) == len(standard_tokens):
                # --- 3. Update the forms with the new normalized tokens ---
                for token, new_form in zip(standard_tokens, normalized_tokens):
                    token["form"] = new_form
            else:
                # Safety check: if normalization merges or deletes words, skip updating forms
                sent_id = sentence.metadata.get("sent_id", "Unknown")
                print(f"Warning: Token count mismatch in '{sent_id}'. "
                      f"Text tokens: {len(normalized_tokens)} | Form tokens: {len(standard_tokens)}. Forms not updated.")
            
            # --- 4. Apply the normalization function to the lemmas ---
            for token in sentence:
                # We normalize only the standard tokens, ignoring empty nodes or MWT headers
                if isinstance(token["id"], int):
                    token["lemma"] = far_tool.normalize_text(token["lemma"])
            
            # Write the updated sentence to the output file
            outfile.write(sentence.serialize())

In [9]:
normalize_conllu_fields("el_gud-ud-train-tokenized.conllu", "el_gud-ud-train-normalized.conllu")
normalize_conllu_fields("el_gud-ud-test-tokenized.conllu", "el_gud-ud-test-normalized.conllu")

KeyboardInterrupt: 